# 5. 滤波器基础

**Fundamental of Filters**

---

本notebook是通信基础系列的第五课，将帮助您理解滤波器的基本概念和设计方法。滤波器是通信系统中最重要的组成部分之一，用于选择性地处理信号频率成分。

## 1. 学习目标

通过本notebook，您将：

- **理解四种滤波器类型**：掌握低通、高通、带通、带阻滤波器的频率特性
- **掌握FIR滤波器设计方法**：理解窗函数法的原理和设计流程
- **理解IIR滤波器基本概念**：了解巴特沃斯、切比雪夫等经典滤波器设计方法
- **能够用Python设计和应用滤波器**：使用scipy.signal进行滤波器设计与分析
- **理解Z变换在离散滤波器分析中的作用**：掌握系统函数的表示方法

## 2. 概念讲解

### 2.1 四种滤波器类型

滤波器是对信号进行频率选择性处理的核心器件。根据频率响应特性，滤波器分为四种基本类型：

| 滤波器类型 | 缩写 | 通过频率 | 抑制频率 | 典型应用 |
|------------|------|----------|----------|----------|
| **低通滤波器** | LPF | 0 ~ fc | fc ~ ∞ | 去除高频噪声、信号平滑 |
| **高通滤波器** | HPF | fc ~ ∞ | 0 ~ fc | 去除直流分量、边缘检测 |
| **带通滤波器** | BPF | fl ~ fh | 0 ~ fl, fh ~ ∞ | 提取特定频段信号 |
| **带阻滤波器** | BSF | 0 ~ fl, fh ~ ∞ | fl ~ fh | 去除特定频率干扰 |

其中 fc 为截止频率，fl 和 fh 为带通/带阻的下限和上限频率。

**理想滤波器 vs 实际滤波器**：
- **理想滤波器**：在通带内增益为1（0dB），阻带内增益为0（-∞dB），具有陡峭的过渡带
- **实际滤波器**：由于物理实现限制，过渡带存在，且通带和阻带都有纹波

### 2.2 FIR滤波器：有限长冲激响应

**FIR滤波器**（Finite Impulse Response）的冲激响应 h[n] 是有限长度的：

$$y[n] = \sum_{k=0}^{N-1} h[k] \cdot x[n-k]$$

其中 N 为滤波器阶数（抽头数）。

**FIR滤波器的主要特点**：

| 特性 | 说明 | 优势 |
|------|------|------|
| **稳定性** | 永远稳定（无反馈支路） | 设计简单，保证稳定 |
| **线性相位** | 可设计为严格线性相位 | 不产生相位失真 |
| **实现简单** | 仅使用抽头系数 | 硬件实现效率高 |
| **群延迟** | 群延迟固定 | 信号时延可精确控制 |

**线性相位的意义**：
若系统具有线性相位，则所有频率成分经历相同的时延：
$$延迟 = \frac{-\phi(\omega)}{\omega} = \tau_g = 常数$$

这对于语音、图像等信号的处理尤为重要，可以避免信号波形的畸变。

**FIR滤波器设计方法**：
1. **窗函数法**：用窗函数截断理想冲激响应
2. **频率采样法**：在频域指定期望响应，通过IDFT得到时域系数
3. **等纹波法**：优化设计使纹波均匀分布

### 2.3 IIR滤波器：无限长冲激响应

**IIR滤波器**（Infinite Impulse Response）具有反馈结构，冲激响应理论上是无限长的：

$$y[n] = \sum_{k=0}^{M} b[k] \cdot x[n-k] - \sum_{k=1}^{N} a[k] \cdot y[n-k]$$

**IIR滤波器的主要特点**：

| 特性 | 说明 | 注意事项 |
|------|------|----------|
| **高效率** | 达到相同规格所需阶数远低于FIR | 计算效率高 |
| **非线性相位** | 相位响应通常是非线性的 | 可能导致相位失真 |
| **可能不稳定** | 反馈回路可能导致不稳定 | 需要极点分析 |
| **经典设计法** | 基于模拟滤波器的变换 | 巴特沃斯、切比雪夫等 |

**常见IIR滤波器类型**：
- **巴特沃斯（Butterworth）**：通带和阻带均无纹波，最大平坦
- **切比雪夫I型（Chebyshev I）**：通带等纹波，阻带单调
- **切比雪夫II型（Chebyshev II）**：通带单调，阻带等纹波
- **椭圆（Elliptic）**：通带和阻带均有纹波，阶数最低

**设计方法**：
IIR滤波器设计通常从模拟滤波器原型出发，通过双线性变换或脉冲不变变换得到数字滤波器。

### 2.4 Z变换简介

**Z变换**是分析离散时间线性时不变系统的重要工具，定义为：

$$H(z) = \mathcal{Z}\{{h[n]\}} = \sum_{n=-\infty}^{\infty} h[n] z^{-n}$$

其中 z 是复变量，z = re^{jω} 表示复平面上的任意点。

**系统函数H(z)与滤波器特性的关系**：

- **极点**：使 H(z) → ∞ 的 z 值，决定系统的频率响应峰值位置
- **零点**：使 H(z) = 0 的 z 值，决定系统的频率响应零点位置

**IIR滤波器的系统函数**：
$$H(z) = \frac{\sum_{k=0}^{M} b[k] z^{-k}}{1 + \sum_{k=1}^{N} a[k] z^{-k}}$$

分子多项式的根是**零点**，分母多项式的根是**极点**。

**稳定性条件**：
对于IIR滤波器，所有极点必须在单位圆内（|z| < 1）。

| 变换 | 连续系统 | 离散系统 |
|------|----------|----------|
| 信号表示 | x(t) | x[n] |
| 系统分析 | 拉普拉斯变换 | **Z变换** |
| 频率响应 | H(jω) = H(s)|_{s=jω} | H(e^{jω}) = H(z)|_{z=e^{jω}} |
| 复变量 | s = σ + jω | z = re^{jω} |

## 3. Python演示

下面我们使用Python来演示滤波器的相关概念和设计方法。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
%matplotlib inline

# 设置中文字体支持
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

print("滤波器演示环境已准备就绪")

### 3.1 四种滤波器类型的频率响应

In [ ]:
def demo_filter_types():
    """
    演示四种基本滤波器类型的频率响应
    使用IIR滤波器设计方法创建低通、高通、带通、带阻滤波器
    
    参数说明：
    - fs: 采样频率 (Hz)
    - fcs: 各滤波器的截止频率参数
    """
    
    fs = 1000  # 采样频率 1000 Hz
    
    # 创建频率轴（用于绘制频率响应）
    w = np.linspace(0.01, 500, 1000)  # 0.01 ~ 500 Hz
    w_norm = w / (fs / 2)  # 归一化频率 [0, 1]
    
    # 设计四种滤波器（使用二阶滤波器作为示例）
    # 低通滤波器：通带 0~150Hz
    b_lpf, a_lpf = signal.butter(4, 150 / (fs / 2), btype='low')
    
    # 高通滤波器：通带 250Hz~
    b_hpf, a_hpf = signal.butter(4, 150 / (fs / 2), btype='high')
    
    # 带通滤波器：通带 100~250Hz
    b_bpf, a_bpf = signal.butter(4, [100 / (fs / 2), 250 / (fs / 2)], btype='band')
    
    # 带阻滤波器：阻带 100~250Hz
    b_bsf, a_bsf = signal.butter(4, [100 / (fs / 2), 250 / (fs / 2)], btype='bandstop')
    
    # 计算各滤波器的频率响应
    w_rad, h_lpf = signal.freqz(b_lpf, a_lpf, worN=1000, fs=fs)
    w_rad, h_hpf = signal.freqz(b_hpf, a_hpf, worN=1000, fs=fs)
    w_rad, h_bpf = signal.freqz(b_bpf, a_bpf, worN=1000, fs=fs)
    w_rad, h_bsf = signal.freqz(b_bsf, a_bsf, worN=1000, fs=fs)
    
    # 转换为分贝（dB）表示
    h_lpf_db = 20 * np.log10(np.abs(h_lpf) + 1e-10)
    h_hpf_db = 20 * np.log10(np.abs(h_hpf) + 1e-10)
    h_bpf_db = 20 * np.log10(np.abs(h_bpf) + 1e-10)
    h_bsf_db = 20 * np.log10(np.abs(h_bsf) + 1e-10)
    
    # 创建图形
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # 低通滤波器
    axes[0, 0].plot(w_rad, h_lpf_db, 'b-', linewidth=2)
    axes[0, 0].axhline(y=-3, color='r', linestyle='--', alpha=0.7, label='-3dB 线')
    axes[0, 0].axvline(x=150, color='g', linestyle='--', alpha=0.7, label='fc=150Hz')
    axes[0, 0].set_ylabel('幅度 (dB)', fontsize=11)
    axes[0, 0].set_title('低通滤波器 (LPF)', fontsize=12)
    axes[0, 0].set_xlim(0, 500)
    axes[0, 0].set_ylim(-80, 5)
    axes[0, 0].legend(loc='lower left')
    axes[0, 0].grid(True, alpha=0.3)
    axes[0, 0].fill_between(w_rad, h_lpf_db, -80, alpha=0.2, color='blue')
    
    # 高通滤波器
    axes[0, 1].plot(w_rad, h_hpf_db, 'b-', linewidth=2)
    axes[0, 1].axhline(y=-3, color='r', linestyle='--', alpha=0.7, label='-3dB 线')
    axes[0, 1].axvline(x=150, color='g', linestyle='--', alpha=0.7, label='fc=150Hz')
    axes[0, 1].set_ylabel('幅度 (dB)', fontsize=11)
    axes[0, 1].set_title('高通滤波器 (HPF)', fontsize=12)
    axes[0, 1].set_xlim(0, 500)
    axes[0, 1].set_ylim(-80, 5)
    axes[0, 1].legend(loc='lower right')
    axes[0, 1].grid(True, alpha=0.3)
    axes[0, 1].fill_between(w_rad, h_hpf_db, -80, alpha=0.2, color='orange')
    
    # 带通滤波器
    axes[1, 0].plot(w_rad, h_bpf_db, 'b-', linewidth=2)
    axes[1, 0].axhline(y=-3, color='r', linestyle='--', alpha=0.7, label='-3dB 线')
    axes[1, 0].axvline(x=100, color='g', linestyle='--', alpha=0.7)
    axes[1, 0].axvline(x=250, color='g', linestyle='--', alpha=0.7)
    axes[1, 0].set_ylabel('幅度 (dB)', fontsize=11)
    axes[1, 0].set_xlabel('频率 (Hz)', fontsize=11)
    axes[1, 0].set_title('带通滤波器 (BPF)', fontsize=12)
    axes[1, 0].set_xlim(0, 500)
    axes[1, 0].set_ylim(-80, 5)
    axes[1, 0].legend(loc='lower left')
    axes[1, 0].grid(True, alpha=0.3)
    axes[1, 0].fill_between(w_rad, h_bpf_db, -80, alpha=0.2, color='green')
    
    # 带阻滤波器
    axes[1, 1].plot(w_rad, h_bsf_db, 'b-', linewidth=2)
    axes[1, 1].axhline(y=-3, color='r', linestyle='--', alpha=0.7, label='-3dB 线')
    axes[1, 1].axvline(x=100, color='g', linestyle='--', alpha=0.7)
    axes[1, 1].axvline(x=250, color='g', linestyle='--', alpha=0.7)
    axes[1, 1].set_ylabel('幅度 (dB)', fontsize=11)
    axes[1, 1].set_xlabel('频率 (Hz)', fontsize=11)
    axes[1, 1].set_title('带阻滤波器 (BSF)', fontsize=12)
    axes[1, 1].set_xlim(0, 500)
    axes[1, 1].set_ylim(-80, 5)
    axes[1, 1].legend(loc='lower right')
    axes[1, 1].grid(True, alpha=0.3)
    axes[1, 1].fill_between(w_rad, h_bsf_db, -80, alpha=0.2, color='red')
    
    plt.tight_layout()
    plt.show()
    
    # 打印滤波器信息
    print("四种滤波器类型频率响应对比")
    print("=" * 60)
    print(f"低通滤波器: 通带 0~150Hz, 阻带 150Hz~")
    print(f"高通滤波器: 通带 150Hz~, 阻带 0~150Hz")
    print(f"带通滤波器: 通带 100~250Hz, 阻带 0~100Hz, 250Hz~")
    print(f"带阻滤波器: 阻带 100~250Hz, 通带 0~100Hz, 250Hz~")
    print("-" * 60)
    print("说明：滤波器阶数均为4阶巴特沃斯滤波器")
    print("      -3dB点附近的频率称为截止频率")
    print("=" * 60)
    
    return

demo_filter_types()

### 3.2 FIR滤波器设计：窗函数法

In [ ]:
def demo_fir_window_design():
    """
    演示FIR滤波器窗函数设计方法
    对比不同窗函数对滤波器性能的影响
    
    参数说明：
    - fs: 采样频率
    - fc: 截止频率
    - N: 滤波器阶数
    - 窗函数类型：矩形窗、汉宁窗、汉明窗、布莱克曼窗
    """
    
    fs = 1000  # 采样频率
    fc = 100   # 截止频率 100Hz
    N = 64     # 滤波器阶数
    
    # 归一化截止频率
    wc = fc / (fs / 2)
    
    # 理想低通滤波器冲激响应（理论计算）
    n = np.arange(N)
    m = N // 2
    h_ideal = np.zeros(N)
    for i in range(N):
        if i == m:
            h_ideal[i] = wc  # 中心点单独处理
        else:
            # 辛格函数：sin(wc*(n-M/2)) / (pi*(n-M/2))
            h_ideal[i] = np.sin(wc * np.pi * (i - m)) / (np.pi * (i - m))
    
    # 不同窗函数
    windows = {
        '矩形窗': np.ones(N),
        '汉宁窗': np.hanning(N),
        '汉明窗': np.hamming(N),
        '布莱克曼窗': np.blackman(N)
    }
    
    # 计算各窗函数的频率响应
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    colors = ['blue', 'green', 'orange', 'red']
    
    for idx, (name, window) in enumerate(windows.items()):
        # 加窗得到FIR滤波器系数
        h_fir = h_ideal * window
        
        # 计算频率响应
        w_rad, h_freq = signal.freqz(h_fir, worN=2000, fs=fs)
        h_freq_db = 20 * np.log10(np.abs(h_freq) + 1e-10)
        
        # 计算理想滤波器的频率响应（参考）
        w_ideal = np.linspace(0, fs/2, 2000)
        h_ideal_freq = np.zeros(2000)
        for i in range(2000):
            if w_ideal[i] <= fc:
                h_ideal_freq[i] = 1.0
            else:
                h_ideal_freq[i] = 0.0
        
        row = idx // 2
        col = idx % 2
        
        # 绘制频率响应
        axes[row, col].plot(w_ideal, h_ideal_freq, 'k--', linewidth=1.5, 
                          label='理想滤波器', alpha=0.7)
        axes[row, col].plot(w_rad, h_freq_db, color=colors[idx], linewidth=2, 
                          label=f'FIR ({name})')
        axes[row, col].axhline(y=-3, color='r', linestyle='--', alpha=0.5)
        axes[row, col].axvline(x=fc, color='g', linestyle='--', alpha=0.5)
        axes[row, col].set_ylabel('幅度 (dB)', fontsize=11)
        axes[row, col].set_xlabel('频率 (Hz)', fontsize=11)
        axes[row, col].set_title(f'{name} 设计的 FIR 滤波器', fontsize=12)
        axes[row, col].set_xlim(0, 300)
        axes[row, col].set_ylim(-80, 5)
        axes[row, col].legend(loc='lower left')
        axes[row, col].grid(True, alpha=0.3)
        
        # 计算过渡带宽度和阻带衰减
        stopband_atten = np.min(h_freq_db[w_rad > fc + 50])
        
        print(f"{name}:")
        print(f"  - 阻带最小衰减: {stopband_atten:.1f} dB")
    
    plt.tight_layout()
    plt.show()
    
    # 对比不同窗函数的性能指标
    print("\n窗函数性能对比表：")
    print("=" * 70)
    print(f"{'窗函数':<12} | {'主瓣宽度':<12} | {'阻带衰减':<12} | {'过渡带':<10}")
    print("-" * 70)
    print(f"{'矩形窗':<12} | {'4π/M':<12} | {'-21dB':<12} | {'0.9π/M':<10}")
    print(f"{'汉宁窗':<12} | {'8π/M':<12} | {'-25dB':<12} | {'3.1π/M':<10}")
    print(f"{'汉明窗':<12} | {'8π/M':<12} | {'-41dB':<12} | {'3.3π/M':<10}")
    print(f"{'布莱克曼窗':<12} | {'12π/M':<12} | {'-57dB':<12} | {'5.5π/M':<10}")
    print("-" * 70)
    print("说明：M为滤波器长度，π对应归一化频率π rad/sample")
    print("      窗函数的主瓣越宽，滤波器过渡带越宽，但阻带衰减越大")
    print("=" * 70)
    
    return

demo_fir_window_design()

### 3.3 IIR滤波器设计：经典滤波器类型对比

In [ ]:
def demo_iir_filter_types():
    """
    演示不同IIR滤波器设计方法的对比
    展示巴特沃斯、切比雪夫I型、切比雪夫II型、椭圆滤波器的特性
    
    参数说明：
    - fs: 采样频率
    - fc: 截止频率
    - rp: 通带纹波 (dB)
    - rs: 阻带衰减 (dB)
    """
    
    fs = 1000  # 采样频率
    fc = 100   # 截止频率
    rp = 1     # 通带纹波 1dB
    rs = 40    # 阻带衰减 40dB
    
    # 计算滤波器阶数（使用椭圆滤波器确定最小阶数）
    Wn = fc / (fs / 2)  # 归一化截止频率
    
    # 各类型IIR滤波器设计
    # 巴特沃斯：最大平坦，通带和阻带均无纹波
    order_b, wb = signal.butterord(Wn, 1.0, rp, rs)
    b_butter, a_butter = signal.butter(order_b, Wn, rp=rp, rs=rs)
    
    # 切比雪夫I型：通带等纹波，阻带单调
    order_c1, wc1 = signal.cheb1ord(Wn, 1.0, rp, rs)
    b_cheb1, a_cheb1 = signal.cheby1(order_c1, rp, Wn)
    
    # 切比雪夫II型：通带单调，阻带等纹波
    order_c2, wc2 = signal.cheb2ord(Wn, 1.0, rp, rs)
    b_cheb2, a_cheb2 = signal.cheby2(order_c2, rp, Wn)
    
    # 椭圆滤波器：通带和阻带均有纹波，阶数最低
    order_e, we = signal.ellipord(Wn, 1.0, rp, rs)
    b_ellip, a_ellip = signal.ellip(order_e, rp, rs, Wn)
    
    # 计算各滤波器频率响应
    w_rad, h_butter = signal.freqz(b_butter, a_butter, worN=2000, fs=fs)
    w_rad, h_cheb1 = signal.freqz(b_cheb1, a_cheb1, worN=2000, fs=fs)
    w_rad, h_cheb2 = signal.freqz(b_cheb2, a_cheb2, worN=2000, fs=fs)
    w_rad, h_ellip = signal.freqz(b_ellip, a_ellip, worN=2000, fs=fs)
    
    # 转换为dB
    h_butter_db = 20 * np.log10(np.abs(h_butter) + 1e-10)
    h_cheb1_db = 20 * np.log10(np.abs(h_cheb1) + 1e-10)
    h_cheb2_db = 20 * np.log10(np.abs(h_cheb2) + 1e-10)
    h_ellip_db = 20 * np.log10(np.abs(h_ellip) + 1e-10)
    
    # 绘制对比图
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    filters_data = [
        ('巴特沃斯滤波器', h_butter_db, 'blue', order_b),
        ('切比雪夫I型滤波器', h_cheb1_db, 'green', order_c1),
        ('切比雪夫II型滤波器', h_cheb2_db, 'orange', order_c2),
        ('椭圆滤波器', h_ellip_db, 'red', order_e)
    ]
    
    for idx, (name, h_db, color, order) in enumerate(filters_data):
        row = idx // 2
        col = idx % 2
        
        axes[row, col].plot(w_rad, h_db, color=color, linewidth=2)
        axes[row, col].axhline(y=-3, color='r', linestyle='--', alpha=0.5, label='-3dB')
        axes[row, col].axhline(y=-40, color='purple', linestyle='--', alpha=0.5, label='-40dB')
        axes[row, col].axvline(x=fc, color='g', linestyle='--', alpha=0.5, label=f'fc={fc}Hz')
        axes[row, col].set_ylabel('幅度 (dB)', fontsize=11)
        axes[row, col].set_xlabel('频率 (Hz)', fontsize=11)
        axes[row, col].set_title(f'{name} (阶数={order})', fontsize=12)
        axes[row, col].set_xlim(0, 300)
        axes[row, col].set_ylim(-80, 5)
        axes[row, col].legend(loc='lower left')
        axes[row, col].grid(True, alpha=0.3)
        axes[row, col].fill_between(w_rad, h_db, -80, where=(h_db < -40), 
                                    alpha=0.2, color='gray')
    
    plt.tight_layout()
    plt.show()
    
    # 打印对比总结
    print("\nIIR滤波器类型对比总结：")
    print("=" * 70)
    print(f"{'滤波器类型':<18} | {'阶数':<6} | {'通带特性':<15} | {'阻带特性':<10}")
    print("-" * 70)
    print(f"{'巴特沃斯':<18} | {order_b:<6} | {'最大平坦':<15} | {'单调下降':<10}")
    print(f"{'切比雪夫I型':<18} | {order_c1:<6} | {'等纹波':<15} | {'单调下降':<10}")
    print(f"{'切比雪夫II型':<18} | {order_c2:<6} | {'单调下降':<15} | {'等纹波':<10}")
    print(f"{'椭圆':<18} | {order_e:<6} | {'等纹波':<15} | {'等纹波':<10}")
    print("-" * 70)
    print("结论：")
    print("- 达到相同规格（rp, rs），椭圆滤波器所需阶数最低")
    print("- 巴特沃斯滤波器相位响应最线性")
    print("- 切比雪夫滤波器在通带或阻带有纹波，但阶数较低")
    print("=" * 70)
    
    return

demo_iir_filter_types()

### 3.4 滤波器的实际应用示例

In [ ]:
def demo_filter_application():
    """
    演示滤波器的实际应用：去除噪声、提取信号
    
    参数说明：
    - fs: 采样频率
    - T: 信号持续时间
    """
    
    fs = 1000  # 采样频率 1000 Hz
    T = 0.5   # 信号持续时间 0.5 秒
    t = np.linspace(0, T, int(fs * T))
    
    # 创建复合信号：50Hz有用信号 + 噪声
    # 有用信号：50Hz正弦波
    signal_50hz = 2 * np.sin(2 * np.pi * 50 * t)
    
    # 添加高频噪声：200Hz和300Hz干扰
    noise = 0.5 * np.sin(2 * np.pi * 200 * t) + 0.3 * np.sin(2 * np.pi * 300 * t)
    
    # 合成带噪声的信号
    x_noisy = signal_50hz + noise
    
    # 设计低通滤波器去除高频噪声
    # 截止频率设为80Hz，小于50Hz信号但大于噪声频率
    fc = 80
    b_lpf, a_lpf = signal.butter(6, fc / (fs / 2), btype='low')
    
    # 使用滤波器处理信号
    x_filtered = signal.filtfilt(b_lpf, a_lpf, x_noisy)
    
    # 计算频谱
    X_noisy = np.fft.fft(x_noisy)
    X_filtered = np.fft.fft(x_filtered)
    freq = np.fft.fftfreq(len(t), 1/fs)
    
    # 可视化
    fig, axes = plt.subplots(3, 2, figsize=(14, 12))
    
    # 原始信号时域
    axes[0, 0].plot(t * 1000, signal_50hz, 'b-', linewidth=1.5)
    axes[0, 0].set_ylabel('幅度', fontsize=11)
    axes[0, 0].set_title('原始50Hz有用信号', fontsize=12)
    axes[0, 0].set_xlabel('时间 (ms)', fontsize=11)
    axes[0, 0].grid(True, alpha=0.3)
    
    # 带噪声信号时域
    axes[0, 1].plot(t * 1000, x_noisy, 'r-', linewidth=1.5)
    axes[0, 1].set_ylabel('幅度', fontsize=11)
    axes[0, 1].set_title('叠加高频噪声后的信号', fontsize=12)
    axes[0, 1].set_xlabel('时间 (ms)', fontsize=11)
    axes[0, 1].grid(True, alpha=0.3)
    
    # 原始信号频谱
    pos_freq_mask = freq > 0
    axes[1, 0].stem(freq[pos_freq_mask] * 1000, 
                   np.abs(X_noisy[pos_freq_mask]) / len(t), 
                   markerfmt=' ', basefmt=' ')
    axes[1, 0].axvline(x=50, color='b', linestyle='--', alpha=0.7, label='50Hz')
    axes[1, 0].axvline(x=200, color='r', linestyle='--', alpha=0.7, label='200Hz噪声')
    axes[1, 0].axvline(x=300, color='g', linestyle='--', alpha=0.7, label='300Hz噪声')
    axes[1, 0].set_ylabel('幅度谱', fontsize=11)
    axes[1, 0].set_title('带噪声信号的频谱', fontsize=12)
    axes[1, 0].set_xlabel('频率 (Hz)', fontsize=11)
    axes[1, 0].set_xlim(0, 400)
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # 滤波后信号频谱
    axes[1, 1].stem(freq[pos_freq_mask] * 1000, 
                   np.abs(X_filtered[pos_freq_mask]) / len(t), 
                   markerfmt=' ', basefmt=' ')
    axes[1, 1].axvline(x=50, color='b', linestyle='--', alpha=0.7)
    axes[1, 1].set_ylabel('幅度谱', fontsize=11)
    axes[1, 1].set_title('滤波后信号的频谱', fontsize=12)
    axes[1, 1].set_xlabel('频率 (Hz)', fontsize=11)
    axes[1, 1].set_xlim(0, 400)
    axes[1, 1].grid(True, alpha=0.3)
    
    # 滤波后信号时域
    axes[2, 0].plot(t * 1000, x_filtered, 'g-', linewidth=1.5)
    axes[2, 0].set_ylabel('幅度', fontsize=11)
    axes[2, 0].set_title(f'低通滤波后信号 (fc={fc}Hz)', fontsize=12)
    axes[2, 0].set_xlabel('时间 (ms)', fontsize=11)
    axes[2, 0].grid(True, alpha=0.3)
    
    # 绘制对比：原始信号 vs 滤波后信号
    axes[2, 1].plot(t * 1000, signal_50hz, 'b-', linewidth=2, label='原始50Hz信号', alpha=0.7)
    axes[2, 1].plot(t * 1000, x_filtered, 'r--', linewidth=2, label='滤波后信号')
    axes[2, 1].set_ylabel('幅度', fontsize=11)
    axes[2, 1].set_title('滤波效果对比', fontsize=12)
    axes[2, 1].set_xlabel('时间 (ms)', fontsize=11)
    axes[2, 1].legend()
    axes[2, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # 计算信噪比改善
    snr_before = 10 * np.log10(np.var(signal_50hz) / np.var(noise))
    snr_after = 10 * np.log10(np.var(signal_50hz) / np.var(x_filtered - signal_50hz))
    
    print("\n滤波器应用效果分析：")
    print("=" * 60)
    print(f"原始信号: 50Hz正弦波")
    print(f"噪声成分: 200Hz + 300Hz正弦波")
    print(f"滤波器规格: 6阶巴特沃斯低通滤波器，截止频率{fc}Hz")
    print("-" * 60)
    print(f"滤波前端噪比(SNR): {snr_before:.2f} dB")
    print(f"滤波后端噪比(SNR): {snr_after:.2f} dB")
    print(f"SNR改善: {snr_after - snr_before:.2f} dB")
    print("=" * 60)
    print("\n结论：")
    print("- 低通滤波器有效抑制了200Hz和300Hz的高频噪声")
    print("- 保留了50Hz有用信号成分")
    print("- filtfilt函数实现零相位滤波，进一步消除相位失真")
    
    return

demo_filter_application()

### 3.5 Z变换与系统函数的极零点分析

In [ ]:
def demo_z_plane_analysis():
    """
    演示Z平面上的极点和零点分析
 极点位置决定系统频率响应特性
    
    参数说明：
    - poles: 极点位置
    - zeros: 零点位置
    """
    
    # 定义一个二阶低通滤波器的系数
    # H(z) = b0 / (1 + a1*z^-1 + a2*z^-2)
    b = [0.2, 0.2, 0.2]  # 分子系数
    a = [1, -0.5, 0.2]   # 分母系数
    
    # 计算极点和零点
    zeros = np.roots(b)
    poles = np.roots(a)
    
    # 创建图形
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Z平面图
    theta = np.linspace(0, 2*np.pi, 100)
    unit_circle_x = np.cos(theta)
    unit_circle_y = np.sin(theta)
    
    axes[0].plot(unit_circle_x, unit_circle_y, 'k-', linewidth=1)
    
    # 绘制极点
    for p in poles:
        axes[0].plot(np.real(p), np.imag(p), 'x', markersize=15, 
                   markeredgewidth=3, color='red', label='极点')
    
    # 绘制零点
    for z in zeros:
        axes[0].plot(np.real(z), np.imag(z), 'o', markersize=10, 
                   markeredgewidth=2, color='blue', markerfacecolor='none', label='零点')
    
    axes[0].axhline(y=0, color='k', linewidth=0.5)
    axes[0].axvline(x=0, color='k', linewidth=0.5)
    axes[0].set_xlabel('实部', fontsize=11)
    axes[0].set_ylabel('虚部', fontsize=11)
    axes[0].set_title('Z平面：极点和零点位置', fontsize=12)
    axes[0].set_xlim(-2, 2)
    axes[0].set_ylim(-2, 2)
    axes[0].set_aspect('equal')
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()
    
    # 添加单位圆标注
    axes[0].text(1.1, 0.1, '单位圆', fontsize=10, color='black')
    
    # 计算并绘制频率响应
    w_rad, h_freq = signal.freqz(b, a, worN=1000)
    h_freq_db = 20 * np.log10(np.abs(h_freq) + 1e-10)
    
    axes[1].plot(w_rad / np.pi * 500, h_freq_db, 'b-', linewidth=2)
    axes[1].axhline(y=-3, color='r', linestyle='--', alpha=0.5)
    axes[1].set_ylabel('幅度 (dB)', fontsize=11)
    axes[1].set_xlabel('频率 (Hz)', fontsize=11)
    axes[1].set_title('频率响应', fontsize=12)
    axes[1].set_xlim(0, 500)
    axes[1].set_ylim(-40, 5)
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # 打印极零点信息
    print("\nZ平面分析：")
    print("=" * 60)
    print(f"零点位置: {zeros}")
    print(f"极点位置: {poles}")
    print("-" * 60)
    print("稳定性分析：")
    print("- 所有极点都在单位圆内 → 系统稳定")
    print("- 极点越靠近单位圆边缘 → 频率响应峰值越尖锐")
    print("- 单位圆上的零点 → 该频率完全被抑制（陷波滤波器）")
    print("=" * 60)
    
    return

demo_z_plane_analysis()

## 4. 原理解析

### 4.1 滤波器设计的关键参数

In [ ]:
def demo_filter_design_parameters():
    """
    解释滤波器设计的关键参数
    截止频率、过渡带宽度、通带纹波、阻带衰减
    """
    
    print("滤波器设计的关键参数")
    print("=" * 70)
    
    print("\n【1】截止频率 (Cutoff Frequency)")
    print("-" * 60)
    print("定义：幅度响应下降到-3dB的频率点")
    print("物理意义：信号功率衰减到一半的频率点")
    print("计算：|H(fc)| = 0.707 × |H(0)| 或 |H(fc)|² = 0.5 × |H(0)|²")
    
    print("\n【2】过渡带宽度 (Transition Band)")
    print("-" * 60)
    print("定义：通带边界到阻带边界之间的频率范围")
    print("重要性：过渡带越窄，滤波器选择性越好，但阶数越高")
    print("实际约束：物理器件无法实现无限陡峭的过渡带")
    
    print("\n【3】通带纹波 (Passband Ripple)")
    print("-" * 60)
    print("定义：通带内幅度响应允许的最大波动范围")
    print("单位：分贝(dB)")
    print("影响：纹波越小，通带内信号保真度越高")
    
    print("\n【4】阻带衰减 (Stopband Attenuation)")
    print("-" * 60)
    print("定义：阻带内幅度响应达到的最小衰减量")
    print("单位：分贝(dB)")
    print("意义：-60dB表示信号衰减到原来的1/1000")
    
    print("\n" + "=" * 70)
    
    # 创建一个具体示例来展示这些参数
    fs = 1000
    
    # 设计一个有明确规格的滤波器
    # 通带: 0~100Hz, 阻带: 200Hz~
    # 通带纹波: 1dB, 阻带衰减: 40dB
    rp = 1    # 通带纹波 1dB
    rs = 40   # 阻带衰减 40dB
    fp = 100  # 通带边界 100Hz
    fs_stop = 200  # 阻带起始 200Hz
    
    # 计算归一化频率
    Wp = fp / (fs / 2)
    Ws = fs_stop / (fs / 2)
    
    # 使用椭圆滤波器设计
    order, Wn = signal.ellipord(Wp, Ws, rp, rs)
    b, a = signal.ellip(order, rp, rs, Wn)
    
    # 计算频率响应
    w_rad, h = signal.freqz(b, a, worN=2000, fs=fs)
    h_db = 20 * np.log10(np.abs(h) + 1e-10)
    
    # 可视化关键参数
    fig, ax = plt.subplots(figsize=(12, 6))
    
    ax.plot(w_rad, h_db, 'b-', linewidth=2)
    
    # 标注通带和阻带边界
    ax.axhline(y=0, color='g', linestyle='--', alpha=0.7, label='通带增益 0dB')
    ax.axhline(y=-rp, color='orange', linestyle='--', alpha=0.7, 
               label=f'通带纹波 -{rp}dB')
    ax.axhline(y=-rs, color='r', linestyle='--', alpha=0.7, 
               label=f'阻带衰减 -{rs}dB')
    
    # 标注截止频率
    idx_fc = np.argmin(np.abs(h_db + 3))
    fc_actual = w_rad[idx_fc]
    ax.axvline(x=fc_actual, color='purple', linestyle='--', alpha=0.7)
    ax.plot(fc_actual, h_db[idx_fc], 'o', markersize=10, color='purple')
    ax.annotate(f'fc≈{fc_actual:.0f}Hz\n(-3dB)', 
                xy=(fc_actual, h_db[idx_fc]), 
                xytext=(fc_actual + 30, h_db[idx_fc] + 10),
                fontsize=10,
                arrowprops=dict(arrowstyle='->', color='purple'))
    
    # 标注通带和阻带区域
    ax.fill_between(w_rad, h_db, -80, where=(w_rad <= fp), 
                    alpha=0.2, color='green', label=f'通带 0~{fp}Hz')
    ax.fill_between(w_rad, h_db, -80, where=(w_rad >= fs_stop), 
                    alpha=0.2, color='red', label=f'阻带 {fs_stop}Hz~')
    ax.fill_between(w_rad, h_db, -80, where=((w_rad > fp) & (w_rad < fs_stop)), 
                    alpha=0.2, color='yellow', label=f'过渡带 {fp}~{fs_stop}Hz')
    
    ax.set_ylabel('幅度 (dB)', fontsize=11)
    ax.set_xlabel('频率 (Hz)', fontsize=11)
    ax.set_title(f'滤波器设计规格与频率响应 (阶数={order})', fontsize=12)
    ax.set_xlim(0, 400)
    ax.set_ylim(-80, 5)
    ax.legend(loc='lower left')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n设计规格：")
    print(f"  - 通带: 0 ~ {fp}Hz")
    print(f"  - 阻带: {fs_stop}Hz ~")
    print(f"  - 通带纹波: {rp}dB")
    print(f"  - 阻带衰减: {rs}dB")
    print(f"  - 实际截止频率: {fc_actual:.1f}Hz")
    print(f"  - 所需阶数: {order}")
    
    return

demo_filter_design_parameters()

### 4.2 FIR与IIR滤波器的对比选择

In [ ]:
def demo_fir_vs_iir():
    """
    对比FIR和IIR滤波器在相同规格下的性能差异
    """
    
    print("FIR与IIR滤波器对比")
    print("=" * 70)
    
    # 相同的设计规格
    fs = 1000
    fc = 100
    rp = 1
    rs = 40
    
    # IIR设计（椭圆滤波器，效率最高）
    Wn = fc / (fs / 2)
    order_iir, _ = signal.ellipord(Wn, 1.01*Wn, rp, rs)
    b_iir, a_iir = signal.ellip(order_iir, rp, rs, Wn)
    
    # FIR设计（窗函数法）
    # 为了达到相同的选择性，FIR需要更高的阶数
    # 经验公式：过渡带宽度约为 4/N（N为阶数）
    # 要达到40dB阻带衰减，使用汉明窗需要过渡带约3.3π/N
    # 设过渡带为 (150-100)/500 = 0.1（归一化）
    # N ≈ 3.3π / 0.1 ≈ 100
    order_fir = 101
    n = np.arange(order_fir)
    m = order_fir // 2
    wc = fc / (fs / 2)
    
    # 理想低通冲激响应
    h_ideal = np.zeros(order_fir)
    for i in range(order_fir):
        if i == m:
            h_ideal[i] = wc
        else:
            h_ideal[i] = np.sin(wc * np.pi * (i - m)) / (np.pi * (i - m))
    
    # 加汉明窗
    h_fir = h_ideal * np.hamming(order_fir)
    
    # 计算频率响应
    w_rad, h_iir = signal.freqz(b_iir, a_iir, worN=2000, fs=fs)
    w_rad, h_fir_resp = signal.freqz(h_fir, worN=2000, fs=fs)
    
    h_iir_db = 20 * np.log10(np.abs(h_iir) + 1e-10)
    h_fir_db = 20 * np.log10(np.abs(h_fir_resp) + 1e-10)
    
    # 绘制对比图
    fig, axes = plt.subplots(2, 1, figsize=(12, 10))
    
    axes[0].plot(w_rad, h_iir_db, 'b-', linewidth=2, label=f'IIR椭圆滤波器 (阶数={order_iir})')
    axes[0].plot(w_rad, h_fir_db, 'r--', linewidth=2, label=f'FIR汉明窗滤波器 (阶数={order_fir})')
    axes[0].axhline(y=-3, color='gray', linestyle='--', alpha=0.5)
    axes[0].axhline(y=-40, color='gray', linestyle='--', alpha=0.5)
    axes[0].axvline(x=fc, color='g', linestyle='--', alpha=0.5)
    axes[0].set_ylabel('幅度 (dB)', fontsize=11)
    axes[0].set_xlabel('频率 (Hz)', fontsize=11)
    axes[0].set_title('频率响应对比', fontsize=12)
    axes[0].set_xlim(0, 400)
    axes[0].set_ylim(-80, 5)
    axes[0].legend(loc='lower left')
    axes[0].grid(True, alpha=0.3)
    
    # 计算群延迟
    _, gd_iir = signal.group_delay((b_iir, a_iir), w=2000, fs=fs)
    _, gd_fir = signal.group_delay((h_fir, [1]), w=2000, fs=fs)
    
    axes[1].plot(w_rad, gd_iir, 'b-', linewidth=2, label='IIR群延迟')
    axes[1].plot(w_rad, gd_fir, 'r--', linewidth=2, label='FIR群延迟')
    axes[1].set_ylabel('群延迟 (样本)', fontsize=11)
    axes[1].set_xlabel('频率 (Hz)', fontsize=11)
    axes[1].set_title('群延迟对比（线性相位特性）', fontsize=12)
    axes[1].set_xlim(0, 400)
    axes[1].legend(loc='upper right')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # 打印对比总结
    print("\n性能对比总结：")
    print("=" * 70)
    print(f"{'特性':<20} | {'IIR滤波器':<20} | {'FIR滤波器':<20}")
    print("-" * 70)
    print(f"{'阶数':<20} | {order_iir:<20} | {order_fir:<20}")
    print(f"{'计算复杂度':<20} | {'低':<20} | {'高':<20}")
    print(f"{'相位特性':<20} | {'非线性':<20} | {'严格线性':<20}")
    print(f"{'稳定性':<20} | {'需检查':<20} | {'永远稳定':<20}")
    print(f"{'群延迟':<20} | {'随频率变化':<20} | {'恒定':<20}")
    print("-" * 70)
    print("\n选择建议：")
    print("- 需要线性相位（如音频、图像处理）→ 选择FIR")
    print("- 对计算效率要求高，阶数受限 → 选择IIR")
    print("- 需要零相位滤波 → 使用filtfilt配合IIR")
    print("- 硬件实现，资源受限 → 选择FIR（无需反馈）")
    print("=" * 70)
    
    return

demo_fir_vs_iir()

## 5. 思考题

1. **滤波器基础**
   - 低通、高通、带通、带阻滤波器分别通过什么频率信号？
   - 什么是截止频率？为什么选择-3dB作为参考点？
   - 画出四种滤波器理想频率响应的大致形状
   
2. **FIR滤波器**
   - FIR滤波器的"有限长冲激响应"是什么意思？为什么是有限的？
   - 为什么FIR滤波器可以做到严格线性相位？这有什么好处？
   - 用窗函数法设计FIR滤波器时，矩形窗、汉宁窗、汉明窗、布莱克曼窗各有什么特点？
   - 为什么窗函数的主瓣越宽，滤波器的过渡带也越宽？
   
3. **IIR滤波器**
   - IIR滤波器的"无限长冲激响应"是如何产生的？
   - 巴特沃斯、切比雪夫、椭圆滤波器各有什么优缺点？
   - 为什么IIR滤波器通常比FIR滤波器阶数低但效率高？
   - 如何判断一个IIR滤波器是否稳定？
   
4. **Z变换**
   - Z变换在离散滤波器分析中的作用是什么？
   - 极点和零点分别如何影响滤波器的频率响应？
   - 为什么稳定IIR滤波器的所有极点必须在单位圆内？
   - 如果一个滤波器在z=1处有一个零点，这意味着什么？
   
5. **滤波器设计**
   - 滤波器设计中的关键参数有哪些？分别代表什么意义？
   - 设计一个音频均衡器需要考虑哪些因素？
   - 在实际应用中，为什么无法实现理想滤波器？
   - FIR和IIR滤波器各适合什么应用场景？
   
6. **综合应用**
   - 在语音通信中，如何使用滤波器来去除噪声？
   - 解释为什么立体声信号处理需要线性相位滤波器？
   - 设计一个能够提取特定电台信号的滤波器，需要考虑哪些参数？
   - 在数字滤波器实现中，filtfilt函数相比lfilter有什么优势？

---

## 总结

本notebook介绍了滤波器的基本概念和设计方法：

- **四种滤波器类型**：低通(LPF)、高通(HPF)、带通(BPF)、带阻(BSF)，根据频率选择特性区分
- **FIR滤波器**：有限长冲激响应，永远稳定，可设计严格线性相位，但阶数较高
- **IIR滤波器**：无限长冲激响应，效率高（阶数低），但相位非线性，需注意稳定性
- **Z变换**：分析离散系统的工具，极零点决定频率响应特性
- **设计参数**：截止频率、过渡带、通带纹波、阻带衰减是滤波器设计的核心参数
- **窗函数法**：FIR滤波器设计的经典方法，不同窗函数提供不同的阻带衰减
- **经典设计法**：IIR滤波器基于巴特沃斯、切比雪夫、椭圆等模拟原型设计

滤波器是信号处理和通信系统的核心组件，选择合适类型的滤波器需要综合考虑应用需求、性能规格和计算资源。